# Shot Sync Quality Review

This notebook is the maintained analysis surface for Sportradar-to-Kinexon shot sync quality. It focuses on the questions that matter operationally:

- how many rows land in each `match_method`
- how large the timing residuals are
- which fixtures create most override and fallback risk
- which rows deserve manual review

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'hbl_raw.duckdb').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing data/hbl_raw.duckdb')

REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / 'data' / 'hbl_raw.duckdb'
con = duckdb.connect(str(DB_PATH), read_only=True)
DB_PATH

In [ ]:
df_sync = con.execute("""
    SELECT
        s.fixture_id,
        COALESCE(m.team_name_home || ' vs ' || m.team_name_away, s.fixture_id) AS fixture_label,
        s.event_id,
        s.match_method,
        s.time_difference_ms,
        s.throw_timestamp_ms,
        s.person_league_id,
        s.detected_shot_id,
        s.attack_type,
        s.sub_type,
        s.success
    FROM shot_events AS s
    LEFT JOIN matches_normalized AS m USING (fixture_id)
""").fetchdf()

df_sync['match_method'] = df_sync['match_method'].fillna('unmatched')
df_sync['has_throw_timestamp'] = df_sync['throw_timestamp_ms'].notna()
df_sync['has_detected_shot'] = df_sync['detected_shot_id'].notna()
df_sync['time_difference_s'] = df_sync['time_difference_ms'] / 1000.0

summary = pd.DataFrame({
    'rows': [len(df_sync)],
    'fixtures': [df_sync['fixture_id'].nunique()],
    'matched_rows': [int(df_sync['has_detected_shot'].sum())],
    'throw_timestamps': [int(df_sync['has_throw_timestamp'].sum())],
    'median_time_diff_ms': [float(df_sync['time_difference_ms'].dropna().median())],
    'p95_time_diff_ms': [float(df_sync['time_difference_ms'].dropna().quantile(0.95))],
})
summary.T.rename(columns={0: 'value'})

In [ ]:
method_counts = (
    df_sync['match_method']
    .value_counts(dropna=False)
    .rename_axis('match_method')
    .reset_index(name='rows')
    .assign(share=lambda frame: frame['rows'] / frame['rows'].sum())
)

palette = {
    'player_time': '#1f77b4',
    'time_priority_override': '#ff7f0e',
    'time_only_fallback': '#d62728',
    'unmatched': '#7f7f7f',
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    method_counts['match_method'],
    method_counts['share'],
    color=[palette.get(value, '#4c78a8') for value in method_counts['match_method']],
)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Share of shot sync outcomes by match method')
ax.set_xlabel('Match method')
ax.set_ylabel('Share of rows')

for bar, rows, share in zip(bars, method_counts['rows'], method_counts['share']):
    ax.text(bar.get_x() + bar.get_width() / 2, share + 0.01, f'{rows:,}', ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()
method_counts

In [ ]:
plot_order = ['player_time', 'time_priority_override', 'time_only_fallback']
fig, axes = plt.subplots(1, len(plot_order), figsize=(16, 4), sharey=True)

for ax, method in zip(axes, plot_order):
    values = df_sync.loc[df_sync['match_method'] == method, 'time_difference_s'].dropna()
    ax.hist(values, bins=30, color=palette[method], alpha=0.85)
    ax.set_title(method)
    ax.set_xlabel('Time difference (seconds)')
    ax.axvline(values.median(), color='black', linestyle='--', linewidth=1, label='median')
    ax.legend(frameon=False)

axes[0].set_ylabel('Rows')
fig.suptitle('Timing residual distributions for matched shot events', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fixture_quality = (
    df_sync.groupby(['fixture_id', 'fixture_label'], dropna=False)
    .agg(
        rows=('event_id', 'count'),
        overrides=('match_method', lambda values: int((values == 'time_priority_override').sum())),
        fallbacks=('match_method', lambda values: int((values == 'time_only_fallback').sum())),
        unmatched=('match_method', lambda values: int((values == 'unmatched').sum())),
        median_time_diff_ms=('time_difference_ms', 'median'),
        p95_time_diff_ms=('time_difference_ms', lambda values: values.dropna().quantile(0.95) if values.notna().any() else np.nan),
    )
    .reset_index()
)
fixture_quality['manual_review_share'] = (fixture_quality['overrides'] + fixture_quality['fallbacks'] + fixture_quality['unmatched']) / fixture_quality['rows']
top_review = fixture_quality.sort_values(['manual_review_share', 'rows'], ascending=[False, False]).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_review['fixture_label'], top_review['manual_review_share'], color='#d62728', alpha=0.85)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Fixtures with the highest manual review share')
ax.set_xlabel('Override + fallback + unmatched share')
ax.set_ylabel('Fixture')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

top_review[['fixture_id', 'fixture_label', 'rows', 'overrides', 'fallbacks', 'unmatched', 'median_time_diff_ms', 'p95_time_diff_ms', 'manual_review_share']]

In [ ]:
review_queue = (
    df_sync.loc[df_sync['match_method'].isin(['time_priority_override', 'time_only_fallback'])]
    .sort_values(['time_difference_ms', 'fixture_label'], ascending=[False, True])
    [['fixture_id', 'fixture_label', 'event_id', 'match_method', 'attack_type', 'sub_type', 'time_difference_ms', 'has_throw_timestamp']]
    .head(25)
)
review_queue